# DC-GBRF Single-Case Diagnosis


Minimal public notebook for one AI4I diagnosis case.
This version keeps only the main LLM diagnosis path.

In [1]:
import importlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from langchain.schema import Document

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import utils.loaders as loaders_module
import utils.interactive_context_builders as interactive_context_builders
import utils.llm_diagnosis_prompting as llm_diagnosis_prompting
import utils.llm_diagnosis_runner as llm_diagnosis_runner
from utils.inference import json_to_matrix
from utils.interactive_graph_fusion import build_kb_matrices_df, build_mode_proximity_matrix
import similarities_utils

loaders_module = importlib.reload(loaders_module)
interactive_context_builders = importlib.reload(interactive_context_builders)
llm_diagnosis_prompting = importlib.reload(llm_diagnosis_prompting)
llm_diagnosis_runner = importlib.reload(llm_diagnosis_runner)

build_embeddings = loaders_module.build_embeddings
build_llm = loaders_module.build_llm
load_knowledge_base = loaders_module.load_knowledge_base
load_vector_store = loaders_module.load_vector_store
validate_and_normalize_llm_params = loaders_module.validate_and_normalize_llm_params
build_supplementary_context_from_json = interactive_context_builders.build_supplementary_context_from_json
run_single_case = llm_diagnosis_runner.run_single_case

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

print("Imports loaded successfully.")

ModuleNotFoundError: No module named 'langchain.schema'

In [6]:
TARGET_STAGE = "hybrid_0.4"
TEST_CASE_FILE_NAME = "observed_graph_UID004410_SPLIT-test_GT-HDF.json"

METHOD_NAME = "DC-GBRF"
PROMPT_RELEASE = "public_default"

LLM_NAME = "GPT-5.2"
LLM_BUILD_NAME = "gpt-5.2"
LLM_REASONING_EFFORT = None
LLM_TEMPERATURE = 0.0
LLM_VERBOSITY = None
LLM_MAX_OUTPUT_TOKENS = 350

SUPPLEMENTARY_VARIANT = "lite"
ALPHA_DK = 0.20
OBSERVED_SUMMARY_TOP_K = 4

SAVE_OUTPUT = False

In [ ]:
PUBLICCODE_ROOT = PROJECT_ROOT.parent
RUN_ROOT = PUBLICCODE_ROOT / "AI4I_reformulation" / "outputs" / "ai4i_benchmark" / TARGET_STAGE
KNOWLEDGE_FILE = RUN_ROOT / "knowledge_base" / f"knowledge_base_{TARGET_STAGE}_prototypical.json"
TEST_GRAPH_DIR = RUN_ROOT / "graphs" / "test"
TEST_CASE_PATH = TEST_GRAPH_DIR / TEST_CASE_FILE_NAME

SUPPLEMENTARY_SOURCE_PATH = PROJECT_ROOT / "assets" / "supplementary" / TARGET_STAGE / f"supplementary_knowledge_{TARGET_STAGE}_lite.json"
SUPPLEMENTARY_VECTOR_STORE_PATH = PROJECT_ROOT / "assets" / "supplementary" / TARGET_STAGE / "supplementary_lite"
SAVE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "single_case_runs"

GRAPH_METHOD_REGISTRY = {
    "Cosine": similarities_utils.calculate_cosine_similarity,
    "Frobenius": similarities_utils.calculate_frobenius_similarity,
    "Jaccard": lambda observation_matrix, kb_matrix: similarities_utils.calculate_jaccard_similarity_topk(observation_matrix, kb_matrix, top_k=2),
    "Pearson": similarities_utils.calculate_pearson_similarity,
}
FUSION_BASE_METHODS = ["Cosine", "Pearson", "Jaccard", "Frobenius"]
GED_DEFAULT_PARAMS = {}

LLM_RUNTIME_PARAMS = validate_and_normalize_llm_params(
    temperature=LLM_TEMPERATURE,
    reasoning_effort=LLM_REASONING_EFFORT,
    verbosity=LLM_VERBOSITY,
    max_output_tokens=LLM_MAX_OUTPUT_TOKENS,
)

display(Markdown(
    "### Current Parameters\n"
    f"- RUN_ROOT: {RUN_ROOT}\n"
    f"- TEST_CASE_FILE_NAME: {TEST_CASE_FILE_NAME}\n"
    f"- METHOD_NAME: {METHOD_NAME}\n"
    f"- PROMPT_RELEASE: {PROMPT_RELEASE}\n"
    f"- LLM_NAME / LLM_BUILD_NAME: {LLM_NAME} / {LLM_BUILD_NAME}\n"
    f"- SUPPLEMENTARY_VARIANT: {SUPPLEMENTARY_VARIANT}\n"
    f"- ALPHA_DK: {ALPHA_DK}\n"
    f"- SAVE_OUTPUT: {SAVE_OUTPUT}"
))

In [8]:
if not TEST_CASE_PATH.exists():
    raise FileNotFoundError(TEST_CASE_PATH)
if not KNOWLEDGE_FILE.exists():
    raise FileNotFoundError(KNOWLEDGE_FILE)
if SUPPLEMENTARY_VARIANT != "none" and not SUPPLEMENTARY_SOURCE_PATH.exists():
    raise FileNotFoundError(SUPPLEMENTARY_SOURCE_PATH)
if SUPPLEMENTARY_VARIANT != "none" and not SUPPLEMENTARY_VECTOR_STORE_PATH.exists():
    raise FileNotFoundError(SUPPLEMENTARY_VECTOR_STORE_PATH)

knowledge_base = load_knowledge_base(KNOWLEDGE_FILE)
failure_modes = knowledge_base["failure_modes"]
first_mode = failure_modes[0]
SENSOR_ORDER = sorted([node["id"] for node in first_mode["nodes"] if node["type"] == "Sensor"])
STATE_ORDER = knowledge_base.get(
    "machine_info",
    {},
).get(
    "state_order",
    [node["id"] for node in first_mode["nodes"] if node["type"] == "State"],
)

knowledge_matrices = {
    mode_data["mode"]: json_to_matrix(mode_data, SENSOR_ORDER, STATE_ORDER)
    for mode_data in failure_modes
}
knowledge_graph_docs = [
    Document(
        page_content=json.dumps(mode_data, ensure_ascii=False),
        metadata={"mode": mode_data["mode"], "stage_name": TARGET_STAGE},
    )
    for mode_data in failure_modes
]

kb_matrices_df = build_kb_matrices_df(
    knowledge_matrices=knowledge_matrices,
    sensor_order=SENSOR_ORDER,
    state_order=STATE_ORDER,
)
mode_order = [doc.metadata["mode"] for doc in knowledge_graph_docs]
_, mode_proximity_matrix = build_mode_proximity_matrix(
    kb_matrices_df=kb_matrices_df,
    mode_order=mode_order,
    source="reference_graph",
    zero_normal_links=True,
    manual_overrides=None,
)

prompt_release_to_version = llm_diagnosis_prompting.PROMPT_RELEASE_TO_VERSION
prompt_version = prompt_release_to_version[PROMPT_RELEASE]
system_prompt = llm_diagnosis_prompting.common_system_prompt_for_version(prompt_version)
observed_summary_version = llm_diagnosis_prompting.OBSERVED_SUMMARY_VERSION
mode_labels = [doc.metadata["mode"] for doc in knowledge_graph_docs]

preview_payload = llm_diagnosis_runner.prepare_single_case_payload(
    test_case_path_resolved=TEST_CASE_PATH,
    prompt_version=prompt_version,
    supplementary_variant="none",
    supplementary_retriever=None,
    method_name=METHOD_NAME,
    alpha_dk=ALPHA_DK,
    observed_summary_top_k=OBSERVED_SUMMARY_TOP_K,
    knowledge_graph_docs=knowledge_graph_docs,
    knowledge_matrices=knowledge_matrices,
    graph_method_registry=GRAPH_METHOD_REGISTRY,
    fusion_base_methods=FUSION_BASE_METHODS,
    sensor_order=SENSOR_ORDER,
    state_order=STATE_ORDER,
    mode_proximity_matrix=mode_proximity_matrix,
    ged_default_params=GED_DEFAULT_PARAMS,
)

preview_ranked_modes = preview_payload.get("retrieval_trace", {}).get("fused_table_after_dk", [])
preview_ranked_modes = [row.get("mode") for row in preview_ranked_modes[:3] if isinstance(row, dict) and row.get("mode")]
supplementary_preview = "Supplementary preview disabled."
if SUPPLEMENTARY_VARIANT != "none":
    supplementary_preview = build_supplementary_context_from_json(
        supplementary_source_path=SUPPLEMENTARY_SOURCE_PATH,
        selected_modes=preview_ranked_modes,
    )

display(Markdown("### Pre-API Check: Observed Evidence Summary"))
print(preview_payload["observed_evidence_summary"] or "(empty)")

display(Markdown("### Pre-API Check: Candidate Context"))
print(preview_payload["candidate_context"] or "(empty)")

display(Markdown("### Pre-API Check: Supplementary Context"))
print(supplementary_preview or "(empty)")

print("Artifacts resolved and pre-API payload built successfully.")

### Pre-API Check: Observed Evidence Summary

- wear_risk -> safe: 1.000
- overstrain_risk -> safe: 0.812
- low_power_risk -> safe: 0.667
- high_power_risk -> safe: 0.667


### Pre-API Check: Candidate Context

## Candidate 1: HDF
- alpha_dk: 0.20
- rrf_score_base: 0.356061
- neighbor_support: 0.159089
- rrf_score_dk: 0.316666

{
  "mode": "HDF",
  "nodes": [
    {
      "id": "thermal_risk",
      "type": "Sensor"
    },
    {
      "id": "low_speed_risk",
      "type": "Sensor"
    },
    {
      "id": "low_power_risk",
      "type": "Sensor"
    },
    {
      "id": "high_power_risk",
      "type": "Sensor"
    },
    {
      "id": "wear_risk",
      "type": "Sensor"
    },
    {
      "id": "overstrain_risk",
      "type": "Sensor"
    },
    {
      "id": "safe",
      "type": "State"
    },
    {
      "id": "caution",
      "type": "State"
    },
    {
      "id": "warning",
      "type": "State"
    },
    {
      "id": "danger",
      "type": "State"
    }
  ],
  "edges": [
    {
      "source": "thermal_risk",
      "target": "safe",
      "probability": 0.2098
    },
    {
      "source": "thermal_risk",
      "target": "caution",
      "probability": 0.2647
    },
    {
      "sou

### Pre-API Check: Supplementary Context

- knowledge_base_name: ref_graph_gen_ai4i_main_v5__hybrid_0.4_supplementary_lite
- system_name: ref_graph_gen_ai4i_main_v5__hybrid_0.4
- description: Compact supplementary knowledge for ref_graph_gen_ai4i_main_v5__hybrid_0.4.
- preview_target_modes: HDF, PWF_high, Normal
- available_top_level_keys: knowledge_base_name, machine_info
- note: current lite JSON stores package-level metadata only; richer supplementary passages are loaded from the vector store during the API run cell.
Artifacts resolved and pre-API payload built successfully.


In [ ]:
embeddings = build_embeddings() if SUPPLEMENTARY_VARIANT != "none" else None
llm = build_llm(
    LLM_BUILD_NAME,
    temperature=LLM_TEMPERATURE,
    reasoning_effort=LLM_REASONING_EFFORT,
    verbosity=LLM_VERBOSITY,
    max_output_tokens=LLM_MAX_OUTPUT_TOKENS,
)
supplementary_retriever = None
if SUPPLEMENTARY_VARIANT != "none":
    supplementary_vector_store = load_vector_store(SUPPLEMENTARY_VECTOR_STORE_PATH, embeddings)
    supplementary_retriever = supplementary_vector_store.as_retriever(search_kwargs={"k": 2})

single_case_result = run_single_case(
    llm=llm,
    system_prompt=system_prompt,
    mode_labels=mode_labels,
    run_root=RUN_ROOT,
    stage_name=TARGET_STAGE,
    test_case_path_resolved=TEST_CASE_PATH,
    method_name=METHOD_NAME,
    llm_name=LLM_NAME,
    llm_build_name=LLM_BUILD_NAME,
    supplementary_variant=SUPPLEMENTARY_VARIANT,
    alpha_dk=ALPHA_DK,
    prompt_version=prompt_version,
    observed_summary_version=observed_summary_version,
    observed_summary_top_k=OBSERVED_SUMMARY_TOP_K,
    supplementary_retriever=supplementary_retriever,
    knowledge_graph_docs=knowledge_graph_docs,
    knowledge_matrices=knowledge_matrices,
    graph_method_registry=GRAPH_METHOD_REGISTRY,
    fusion_base_methods=FUSION_BASE_METHODS,
    sensor_order=SENSOR_ORDER,
    state_order=STATE_ORDER,
    mode_proximity_matrix=mode_proximity_matrix,
    ged_default_params=GED_DEFAULT_PARAMS,
)

display(Markdown("### Observed Evidence Summary"))
print(single_case_result.get("observed_evidence_summary") or "(empty)")

display(Markdown("### Candidate Context"))
print(single_case_result.get("candidate_context") or "(empty)")

display(Markdown("### Raw LLM Output"))
print(single_case_result.get("raw_answer") or "(empty)")

if SAVE_OUTPUT:
    SAVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_path = SAVE_OUTPUT_DIR / f"raw_llm_output_{METHOD_NAME.replace(' ', '_')}_{TEST_CASE_PATH.stem}.md"
    with open(save_path, "w", encoding="utf-8") as file_obj:
        file_obj.write(single_case_result.get("raw_answer") or "")
    print("Saved raw LLM output to:", save_path)